In [1]:
import os, json, random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, AllChem, DataStructs
import selfies
from tqdm import tqdm
from rdkit.Chem import PandasTools

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [3]:
from itertools import product

# Load Lewis acid and base datasets
acid = pd.read_csv('acid.csv')
acid = acid.sample(frac=0.1, random_state=42)  # Subsample for efficiency
base = pd.read_csv('zinc.csv')

# Ensure 'smiles' column exists
assert 'smiles' in acid.columns, "Acid dataset must contain a 'smiles' column"
assert 'smiles' in base.columns, "Base dataset must contain a 'smiles' column"

# Convert SMILES to RDKit Mol objects (optional if you use Mol-based features)
PandasTools.AddMoleculeColumnToFrame(acid, smilesCol='smiles')
PandasTools.AddMoleculeColumnToFrame(base, smilesCol='smiles')

# Generate all possible FLP pairs
def generate_flp_pairs(df_acid, df_base, acid_col='smiles', base_col='smiles'):
    acids = df_acid[acid_col].unique()
    bases = df_base[base_col].unique()
    flp_data = [{'flp_smiles': f"{b}.{a}", 'type': 'inter'} for a, b in product(acids, bases)]
    return pd.DataFrame(flp_data)

# Create the FLP pair dataframe
flp_all_df = generate_flp_pairs(acid, base)
smiles_list = flp_all_df['flp_smiles'].dropna().tolist()

In [4]:
def generate_unique_pairs(base_smiles_list, acid_smiles_list, max_pairs):
    seen = set()
    unique_pairs = []

    while len(unique_pairs) < max_pairs:
        b = random.choice(base_smiles_list)
        a = random.choice(acid_smiles_list)
        pair_key = (b, a)

        if pair_key not in seen:
            seen.add(pair_key)
            unique_pairs.append({'flp_smiles': f"{b}.{a}", 'type': 'inter'})

        # Optional: break early if we've exhausted all possible unique pairs
        if len(seen) >= len(base_smiles_list) * len(acid_smiles_list):
            break

    return unique_pairs

In [5]:
import json
from pathlib import Path
from torch.utils.data import Dataset
from selfies import encoder, split_selfies

# Dataset class for SELFIES tokenization
class MolecularSELFIESDataset(Dataset):
    def __init__(self, smiles_list, max_seq_len=60, save_path="tokenized_dataset.pt"):
        self.max_seq_len = max_seq_len
        self.save_path = save_path
        self.smiles = smiles_list

        # Load preprocessed data and vocab if available
        if Path(save_path).exists() and Path("vocab.json").exists():
            print(f"✅ Loading preprocessed token tensors from {save_path}...")
            self.tokenized_tensors = torch.load(save_path)

            with open("vocab.json", "r") as f:
                vocab_data = json.load(f)
            self.token_to_idx = vocab_data["token_to_idx"]
            self.idx_to_token = {int(k): v for k, v in vocab_data["idx_to_token"].items()}
            self.vocab_size = max(self.token_to_idx.values()) + 1
            return

        # Build vocabulary
        print("📖 Building vocabulary...")
        sample_tokens = []
        for smi in self.smiles[:50000]:
            try:
                tokens = list(split_selfies(encoder(smi)))
                sample_tokens.extend(tokens)
            except:
                continue

        unique_tokens = sorted(set(sample_tokens))
        self.token_to_idx = {token: idx + 1 for idx, token in enumerate(unique_tokens)}  # 0 = PAD
        self.idx_to_token = {idx: token for token, idx in self.token_to_idx.items()}
        self.vocab_size = max(self.token_to_idx.values()) + 1

        # Save vocab
        vocab_data = {
            "token_to_idx": self.token_to_idx,
            "idx_to_token": self.idx_to_token
        }
        with open("vocab.json", "w") as f:
            json.dump(vocab_data, f)

        # Tokenize and pad
        print("✏️ Tokenizing SELFIES...")
        self.tokenized_tensors = []
        for smi in tqdm(self.smiles, desc="Tokenizing"):
            try:
                selfies_str = encoder(smi)
                tokens = list(split_selfies(selfies_str))
                token_indices = [self.token_to_idx.get(t, 0) for t in tokens]
                padded = token_indices + [0] * (self.max_seq_len - len(token_indices))
                padded = padded[:self.max_seq_len]
                self.tokenized_tensors.append(torch.tensor(padded, dtype=torch.long))
            except:
                self.tokenized_tensors.append(torch.zeros(self.max_seq_len, dtype=torch.long))

        # Save tokenized dataset
        print(f"💾 Saving tokenized dataset to {save_path}...")
        torch.save(self.tokenized_tensors, save_path)

    def __getitem__(self, idx):
        return self.tokenized_tensors[idx]

    def __len__(self):
        return len(self.tokenized_tensors)
# Load dataset using the tokenized form
dataset = MolecularSELFIESDataset(smiles_list)


✅ Loading preprocessed token tensors from tokenized_dataset.pt...


KeyboardInterrupt: 

In [ ]:

class Generator(nn.Module):
    def __init__(self, latent_dim, embed_dim, hidden_dim, seq_len, vocab_size):
        super().__init__()
        self.seq_len = seq_len
        self.fc = nn.Linear(latent_dim, hidden_dim)
        self.rnn = nn.GRU(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        self.embed_out = nn.Linear(hidden_dim, vocab_size)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.vocab_size = vocab_size
        self.latent_dim = latent_dim

    def forward(self, z):
        batch_size = z.size(0)
        h0 = torch.tanh(self.fc(z)).unsqueeze(0)  # initial hidden state
        inputs = torch.zeros(batch_size, self.seq_len, self.token_embedding.embedding_dim, device=z.device)
        output, _ = self.rnn(inputs, h0)
        logits = self.embed_out(output)
        return logits  # shape: [B, L, V]


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}", flush=True)

In [ ]:
# Set these based on your dataset
latent_dim = 128
embed_dim = 128
hidden_dim = 256
seq_len = dataset.max_seq_len
vocab_size = dataset.vocab_size

# Initialize generator
generator = Generator(latent_dim, embed_dim, hidden_dim, seq_len, vocab_size).to(device)

# Load checkpoint
checkpoint_path = "output/checkpoint.pth"
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    generator.load_state_dict(checkpoint["G"])
    print("✅ Generator model loaded from checkpoint.")
else:
    print("⚠️ No checkpoint found. Generator initialized randomly.")

In [ ]:

class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# Compute input dimension: one-hot size * sequence length
data_dim = dataset.max_seq_len * dataset.vocab_size

# Initialize discriminator
discriminator = Discriminator(input_dim=data_dim).to(device)

# Load from checkpoint
checkpoint_path = "output/checkpoint.pth"
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    discriminator.load_state_dict(checkpoint["D"])
    print("✅ Discriminator model loaded from checkpoint.")
else:
    print("⚠️ No checkpoint found. Discriminator initialized randomly.")

In [ ]:

# --- Helper: reconstruct SELFIES string from generator output ---
def reconstruct_selfies(gen_output, max_seq_len, vocab_size, idx_to_token):
    """
    Convert generator output (flattened logits) into SELFIES string.
    """
    reshaped = gen_output.reshape(max_seq_len, vocab_size)
    token_indices = np.argmax(reshaped, axis=1)
    token_indices = [idx for idx in token_indices if idx != 0]  # remove PAD
    tokens = [idx_to_token.get(idx, '') for idx in token_indices]
    selfies_str = "".join(tokens)
    return selfies_str

# --- Sample a batch from generator ---
generator.eval()
z = torch.randn(1, generator.latent_dim).to(device)  # batch size = 1
with torch.no_grad():
    logits = generator(z).cpu().numpy()  # shape: [1, L, V]
    output_vector = logits[0]  # shape: [L, V]

# --- Convert to SELFIES then SMILES ---
selfies_str = reconstruct_selfies(output_vector, dataset.max_seq_len, dataset.vocab_size, dataset.idx_to_token)
try:
    smiles_str = selfies.decoder(selfies_str)
except:
    smiles_str = None

print("Generated SELFIES:", selfies_str)
print("Decoded SMILES:", smiles_str)

In [ ]:
# Set up
device = torch.device("cpu")
output_dir = "output"
checkpoint_path = os.path.join(output_dir, "checkpoint.pth")
metrics_path = os.path.join(output_dir, "training_metrics.csv")

# Reconstruct models with correct dimensions
latent_dim = 128
latent_dim = 128
hidden_dim = 256
embed_dim = 128
seq_len = dataset.max_seq_len
vocab_size = dataset.vocab_size

G = Generator(latent_dim, embed_dim, hidden_dim, seq_len, vocab_size).to(device)
D = Discriminator(data_dim).to(device)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
G.load_state_dict(checkpoint["G"])
D.load_state_dict(checkpoint["D"])
print(f"✅ Loaded checkpoint from epoch {checkpoint['epoch']}")

# Load metrics
metrics = pd.read_csv(metrics_path)
print("Last 5 epochs of generator loss:")
print(metrics.tail(5))

# Optional: Plot
plt.figure()
plt.plot(metrics.index, metrics["loss_G"], label="Generator")
plt.plot(metrics.index, metrics["loss_D"], label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("WGAN Training Losses")
plt.show()

In [ ]:
plt.figure()
plt.plot(metrics.index, metrics["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss over Iterations")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

LEWIS_BASE_SMARTS = [
    "[#7;H2,H1;!$(NC=O)]",  # primary/secondary amines
    "[O-]",                 # negative oxygen
    "[nH]",                 # aromatic N-H
    "[#8;H1]"               # alcohol OH
]

LEWIS_ACID_SMARTS = [
    "[B]",      # boron
    "[P+]",     # phosphonium
    "[Al]",     # aluminium
    "[Sn]",     # tin
    "[Si]"      # silicon
]

def contains_substructure(mol, smarts_list):
    for smarts in smarts_list:
        pattern = Chem.MolFromSmarts(smarts)
        if mol.HasSubstructMatch(pattern):
            return True
    return False

def reward_contains_flp(smiles: str) -> float:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0.0
    has_base = contains_substructure(mol, LEWIS_BASE_SMARTS)
    has_acid = contains_substructure(mol, LEWIS_ACID_SMARTS)
    return 1.0 if has_base and has_acid else 0.0

In [ ]:
REDUCIBLE_SMARTS = [
    "[C]=[O]",          # carbonyl
    "[C]=[N]",          # imine
    "[N+](=O)[O-]",     # nitro
    "[C]=[C]",          # alkene
    "[N]=[N+]=[N-]"     # azide
]

def reward_hydrogenation_potential(smiles: str) -> float:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0.0
    matches = 0
    for smarts in REDUCIBLE_SMARTS:
        patt = Chem.MolFromSmarts(smarts)
        matches += len(mol.GetSubstructMatches(patt))
    return min(matches, 3) / 3.0  # capped at 1.0

In [ ]:
def combined_flp_reward(smiles: str) -> float:
    base_acid = reward_contains_flp(smiles)
    hydrogenation = reward_hydrogenation_potential(smiles)
    return 0.7 * base_acid + 0.3 * hydrogenation  # adjustable weights

In [ ]:
if smiles_str:
    score = combined_flp_reward(smiles_str)
    print("FLP score:", score)
else:
    print("Invalid SELFIES → SMILES decoding.")

In [ ]:
n_samples = 10000  # Number of molecules to generate
G.eval()  # Set the generator to evaluation mode

generated_smiles_list = set()

with torch.no_grad():
    # Generate a batch of latent vectors
    z = torch.randn(n_samples, latent_dim)
    # Generate samples from the generator
    gen_samples = G(z).detach().cpu().numpy()
    
    # For each generated sample, reconstruct the SELFIES and convert to SMILES
    for gen_sample in gen_samples:
        selfies_str = reconstruct_selfies(gen_sample, dataset.max_seq_len, dataset.vocab_size, dataset.idx_to_token)
        smiles_str = selfies.decoder(selfies_str)
        if smiles_str != "":
            generated_smiles_list.add(smiles_str)

# Print all generated SMILES
for i, smi in enumerate(generated_smiles_list, 1):
    print(f"Molecule {i}: {smi}")

In [ ]:

def validate_smiles(smiles_str):
    """
    Validate a SMILES string using RDKit.
    
    Args:
        smiles_str (str): The SMILES string to validate.
        
    Returns:
        bool: True if the SMILES is valid, False otherwise.
    """
    try:
        # Convert the SMILES string into an RDKit molecule
        mol = Chem.MolFromSmiles(smiles_str)
        if mol is None:
            return False
        
        # Optional: sanitize the molecule to catch any additional issues
        Chem.SanitizeMol(mol)
        return True
    except Exception as e:
        # If any exception occurs, the SMILES is considered invalid
        return False
for i, smi in enumerate(generated_smiles_list, 1):
    print(smi)
    if validate_smiles(smi):
        print("The generated SMILES is valid!")
    else:
        print("The generated SMILES is invalid.")

In [ ]:
from rdkit.Chem import Draw

def visualize_smiles(smiles_str):
    """
    Convert a SMILES string to an RDKit molecule and visualize it.
    
    Args:
        smiles_str (str): The SMILES string to visualize.
    """
    # Convert SMILES to an RDKit molecule
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        print("Invalid SMILES string.")
        return
    
    # Option 1: Using MolToImage to create a PIL image and display it with matplotlib
    img = Draw.MolToImage(mol, size=(300, 300))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Molecular Graph")
    plt.show()

#for i, smi in enumerate(generated_smiles_list, 1):
#    print(smi)
#    visualize_smiles(smi)

In [ ]:
from rdkit.Chem import AllChem, DataStructs

def measure_uniqueness(smiles_list):
    """
    Calculates the fraction of unique SMILES.
    
    Args:
        smiles_list (list of str): A list of SMILES strings.
    
    Returns:
        float: The fraction of unique SMILES in the list.
    """
    # Filter out invalid SMILES (None after parsing)
    valid_smiles = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            valid_smiles.append(Chem.MolToSmiles(mol, isomericSmiles=True))
    # Convert to set to remove duplicates
    unique_smiles = set(valid_smiles)
    if len(valid_smiles) == 0:
        return 0.0
    return len(unique_smiles) / len(valid_smiles) * 100

def measure_diversity(smiles_list, radius=2, nBits=1024):
    """
    Computes the average pairwise Tanimoto distance (1 - similarity) among molecules.
    A higher average distance indicates higher diversity.
    
    Args:
        smiles_list (list of str): A list of SMILES strings.
        radius (int): Radius parameter for Morgan fingerprint.
        nBits (int): Size of the fingerprint bit vector.
    
    Returns:
        float: Average pairwise Tanimoto distance (1 - similarity).
    """
    # Generate fingerprints for all valid SMILES
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
            fps.append(fp)
    
    if len(fps) < 2:
        # Not enough molecules to compare
        return 0.0
    
    # Compute pairwise Tanimoto similarities
    similarities = []
    for i in range(len(fps)):
        for j in range(i + 1, len(fps)):
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            similarities.append(sim)
    
    # Convert similarity to distance = 1 - similarity
    distances = [1 - sim for sim in similarities]
    
    # Return the average distance
    return np.mean(distances)

def is_valid_smiles(smiles):
    return Chem.MolFromSmiles(smiles) is not None


valid_molecules = [s for s in generated_smiles_list if is_valid_smiles(s)]
validity = len(valid_molecules) / n_samples * 100
print(f"Validity: {validity:.2f}%")


training_set = set(smiles_list)
novel_molecules = [s for s in valid_molecules if s not in training_set]
novelty = len(novel_molecules) / len(valid_molecules) * 100
print(f"Novelty: {novelty:.2f}%")

uniqueness = measure_uniqueness(generated_smiles_list)
diversity = measure_diversity(generated_smiles_list, radius=2, nBits=1024)

print(f"Uniqueness: {uniqueness:.2f}%")
print(f"Average Pairwise Tanimoto Distance: {diversity:.3f}")

In [ ]:
full_acid = pd.read_csv('acid.csv')
flp_full_df = generate_flp_pairs(full_acid, base)
full_smiles_list = flp_full_df['flp_smiles'].dropna().tolist()

full_set = set(full_smiles_list)
novel_molecules = [s for s in valid_molecules if s not in full_set]
novelty = len(novel_molecules) / len(valid_molecules) * 100
print(f"Novelty (for full dataset not just training set): {novelty:.2f}%")

In [ ]:
gen_df = pd.DataFrame(generated_smiles_list, columns=['smiles'])

In [ ]:
gen_df

In [ ]:
from rdkit.Chem.rdPartialCharges import ComputeGasteigerCharges

def extract_flp_features(flp_smiles):
    features = {
        'charge_diff': None,
        'tpsa': None,
        'rotatable_bonds': None,
        'num_rings': None
    }

    try:
        if '.' in flp_smiles:
            # Intermolecular FLP pair: split into base + acid
            base_smiles, acid_smiles = flp_smiles.split('.')
            base_mol = Chem.MolFromSmiles(base_smiles)
            acid_mol = Chem.MolFromSmiles(acid_smiles)
            if base_mol is None or acid_mol is None:
                return features

            # Add Hs + charges
            base = Chem.AddHs(base_mol)
            acid = Chem.AddHs(acid_mol)
            ComputeGasteigerCharges(base)
            ComputeGasteigerCharges(acid)

            base_charges = [float(atom.GetProp('_GasteigerCharge')) for atom in base.GetAtoms()]
            acid_charges = [float(atom.GetProp('_GasteigerCharge')) for atom in acid.GetAtoms()]
            features['charge_diff'] = abs(max(base_charges) - min(acid_charges))

            # Sum properties
            features['tpsa'] = Descriptors.TPSA(base) + Descriptors.TPSA(acid)
            features['rotatable_bonds'] = Descriptors.NumRotatableBonds(base) + Descriptors.NumRotatableBonds(acid)
            features['num_rings'] = base.GetRingInfo().NumRings() + acid.GetRingInfo().NumRings()

        else:
            # Intramolecular FLP: single molecule
            mol = Chem.MolFromSmiles(flp_smiles)
            if mol is None:
                return features

            mol = Chem.AddHs(mol)
            ComputeGasteigerCharges(mol)
            charges = [float(atom.GetProp('_GasteigerCharge')) for atom in mol.GetAtoms()]
            # Charge diff: max minus min in same molecule
            features['charge_diff'] = abs(max(charges) - min(charges))

            # Single-molecule descriptors
            features['tpsa'] = Descriptors.TPSA(mol)
            features['rotatable_bonds'] = Descriptors.NumRotatableBonds(mol)
            features['num_rings'] = mol.GetRingInfo().NumRings()

    except Exception as e:
        print("⚠️ Feature extraction failed for:", flp_smiles, "| Error:", e)

    return features

In [ ]:
gen_feature_dicts = gen_df["smiles"].apply(extract_flp_features)
gen_features_df = pd.DataFrame(gen_feature_dicts.tolist())

In [ ]:
gen_features_df

In [ ]:
flp_all_features_df = pd.concat([gen_df, gen_features_df], axis=1)

In [ ]:
flp_all_features_df

In [ ]:
# Rank molecules based on criteria:
# 1. High charge_diff
# 2. Moderate TPSA (30–70 ideal)
# 3. Moderate rotatable bonds (2–6 ideal)
# 4. Prefer 1–2 rings

# Create a scoring system based on the above
def score_molecule(row):
    score = 0
    if row['tpsa'] >= 30 and row['tpsa'] <= 70:
        score += 1
    if row['rotatable_bonds'] >= 2 and row['rotatable_bonds'] <= 6:
        score += 1
    if row['num_rings'] >= 1 and row['num_rings'] <= 2:
        score += 1
    return score

# Apply scoring and sort
flp_all_features_df['score'] = flp_all_features_df.apply(score_molecule, axis=1)
df_sorted = flp_all_features_df.sort_values(by=['score', 'charge_diff'], ascending=[False, False])

# Select top candidates
best = df_sorted.loc[df_sorted["score"] >= 3 ]
top_candidates = best.reset_index(drop=True)

In [ ]:
top_candidates

In [ ]:
top_candidates.to_csv("output.csv", index=False)

In [ ]:
dynamic = True

if dynamic:

    # Plot Learning Rates
    plt.figure(figsize=(8, 4))
    plt.plot(metrics.index, metrics["lr_G"], label="Generator LR", color='blue')
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.title("Learning Rate Over Epochs")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(metrics.index, metrics["lr_D"], label="Discriminator LR", color='orange')
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.title("Learning Rate Over Epochs")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Plot Reward Components
plt.figure(figsize=(10, 6))
plt.plot(metrics.index, metrics["reward_total"], label="Total Reward", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Reward")
plt.title("Reward Components Over Epochs")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics.index, metrics["reward_frustration"], label="Frustration")
plt.xlabel("Epoch")
plt.ylabel("Reward")
plt.title("Reward Components Over Epochs")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics.index, metrics["reward_homo_lumo"], label="HOMO-LUMO")
plt.xlabel("Epoch")
plt.ylabel("Reward")
plt.title("Reward Components Over Epochs")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics.index, metrics["reward_acid_base"], label="Acid/Base")
plt.xlabel("Epoch")
plt.ylabel("Reward")
plt.title("Reward Components Over Epochs")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

import rdkit
from rdkit.Chem import Descriptors, rdMolDescriptors, QED
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem
from scipy.spatial.distance import pdist, squareform
from collections import Counter
import time
from rdkit import Chem
from rdkit.Chem import RDConfig
import os
import sys
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
import sascorer

# Optional: ensure the sascorer module is available (from RDKit contrib)
# You may need to download sascorer.py from https://github.com/rdkit/rdkit/blob/master/Contrib/SA_Score/sascorer.py
#try:
#    import sascorer
#except ImportError:
#    sascorer = None

def compute_validity(smiles_list):
    """Return fraction of valid molecules."""
    valid = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        valid.append(mol is not None)
    return np.mean(valid), valid

def compute_uniqueness(smiles_list, valid_mask):
    """Return fraction of unique valid molecules."""
    valid_smiles = [s for s, v in zip(smiles_list, valid_mask) if v]
    return len(set(valid_smiles)) / len(valid_smiles) if valid_smiles else 0

def compute_novelty(smiles_list, valid_mask, train_smiles_set):
    """Fraction of valid molecules not in training set."""
    valid_smiles = [s for s, v in zip(smiles_list, valid_mask) if v]
    novel = [s for s in valid_smiles if s not in train_smiles_set]
    return len(novel) / len(valid_smiles) if valid_smiles else 0

def compute_diversity(smiles_list, valid_mask):
    """Average pairwise Tanimoto distance among valid molecules."""
    fps = []
    for smi, valid in zip(smiles_list, valid_mask):
        if valid:
            mol = Chem.MolFromSmiles(smi)
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
            fps.append(fp)
    if len(fps) < 2:
        return 0
    # Compute pairwise Tanimoto similarity then convert to distance
    n = len(fps)
    sims = []
    for i in range(n):
        for j in range(i+1, n):
            sims.append(AllChem.DataStructs.TanimotoSimilarity(fps[i], fps[j]))
    return 1 - np.mean(sims)

def compute_physchem(smiles_list, valid_mask):
    """Compute MW, logP, TPSA, QED, SA score."""
    results = []
    for smi, valid in zip(smiles_list, valid_mask):
        if not valid:
            results.append((None,)*5)
            continue
        mol = Chem.MolFromSmiles(smi)
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        tpsa = rdMolDescriptors.CalcTPSA(mol)
        qed = QED.qed(mol)
        sa = sascorer.calculateScore(mol) if sascorer else None
        results.append((mw, logp, tpsa, qed, sa))
    df = pd.DataFrame(results, columns=['MW', 'LogP', 'TPSA', 'QED', 'SA'])
    return df

def compute_scaffold_diversity(smiles_list, valid_mask):
    """Fraction of unique Bemis-Murcko scaffolds."""
    scaffolds = []
    for smi, valid in zip(smiles_list, valid_mask):
        if valid:
            mol = Chem.MolFromSmiles(smi)
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
            scaffolds.append(scaffold)
    return len(set(scaffolds)) / len(scaffolds) if scaffolds else 0

def nearest_neighbor_similarity(smiles_list, valid_mask, train_smiles_list):
    """Compute mean nearest-neighbor Tanimoto similarity to training set."""
    train_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, nBits=2048)
                 for s in train_smiles_list]
    nn_sims = []
    for smi, valid in zip(smiles_list, valid_mask):
        if not valid:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smi), 2, nBits=2048)
        sims = [AllChem.DataStructs.TanimotoSimilarity(fp, tfp) for tfp in train_fps]
        nn_sims.append(max(sims))
    return np.mean(nn_sims) if nn_sims else 0

def reward_statistics(rewards):
    """Return mean, variance, skewness of rewards."""
    return pd.Series(rewards).agg(['mean', 'var', 'skew'])

def time_to_threshold(rewards, threshold):
    """Epoch index when reward first exceeds threshold."""
    for i, r in enumerate(rewards):
        if r >= threshold:
            return i
    return None

# Placeholder for application-specific metrics:
def compute_homo_lumo_gap(smiles_list):
    """Compute HOMO-LUMO gap via external QM or ML model (placeholder)."""
    # Implement using psi4/PySCF or pretrained ML surrogate.
    return [None]*len(smiles_list)

def compute_lewis_descriptors(smiles_list):
    """Compute Lewis acidity/basicity descriptors (placeholder)."""
    return [None]*len(smiles_list)

# Example wrapper function:
def compute_all_metrics(generated_smiles_list, smiles_list):
    train_set = set(smiles_list)
    validity_score, valid_mask = compute_validity(generated_smiles_list)
    uniq_score = compute_uniqueness(generated_smiles_list, valid_mask)
    nov_score = compute_novelty(generated_smiles_list, valid_mask, train_set)
    div_score = compute_diversity(generated_smiles_list, valid_mask)
    physchem_df = compute_physchem(generated_smiles_list, valid_mask)
    scaffold_div = compute_scaffold_diversity(generated_smiles_list, valid_mask)
    nn_sim = nearest_neighbor_similarity(generated_smiles_list, valid_mask, smiles_list)
    # Placeholder results for advanced metrics
    homo_lumo = compute_homo_lumo_gap(generated_smiles_list)
    lewis_desc = compute_lewis_descriptors(generated_smiles_list)

    # Aggregate into a dictionary
    upd_metrics = {
        'validity': validity_score,
        'uniqueness': uniq_score,
        'novelty': nov_score,
        'diversity': div_score,
        'scaffold_diversity': scaffold_div,        'nn_similarity': nn_sim,
    }
    # Merge physchem into metrics
    for col in physchem_df.columns:
        upd_metrics[f'{col}_mean'] = physchem_df[col].mean()
        upd_metrics[f'{col}_std'] = physchem_df[col].std()
    # Return numeric metrics and full data for custom metrics
    return upd_metrics, physchem_df, homo_lumo, lewis_desc

# Example usage:
if __name__ == "__main__":
    # Replace with your data
    # validity_score, valid_mask = compute_validity(generated_smiles_list)
    # physchem_df = compute_physchem(generated_smiles_list, valid_mask)
    # for col in physchem_df.columns:
    #    print(f'{col}_mean', physchem_df[col].mean())
    #    print(f'{col}_std', physchem_df[col].std())

    upd_metrics, physchem, homo_lumo, lewis_desc = compute_all_metrics(generated_smiles_list, smiles_list)
    print(pd.Series(upd_metrics))
    print(physchem.head())

In [ ]:
upd_metrics

In [ ]:

# Assuming you have already run:
# metrics, physchem, homo_lumo_list, lewis_desc_list, rewards = compute_all_metrics(...)

# 1. Bar chart of global generative metrics
global_metrics = ['validity', 'uniqueness', 'novelty', 'diversity', 'scaffold_diversity', 'nn_similarity']
values = [upd_metrics[m] for m in global_metrics]

plt.figure()
#plt.bar(global_metrics, values)
plt.ylabel('Score')
plt.title('Global Generative Metrics')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 2. Histogram for each physicochemical property
for col in physchem.columns:
    if physchem[col].dropna().empty:
        continue
    plt.figure()
    plt.hist(physchem[col].dropna(), bins=30)
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.show()

# 3. Scatter plot QED vs SA (if available)
if 'QED' in physchem.columns and 'SA' in physchem.columns:
    df = physchem.dropna(subset=['QED', 'SA'])
    if not df.empty:
        plt.figure()
        plt.scatter(df['QED'], df['SA'])
        plt.xlabel('QED')
        plt.ylabel('SA')
        plt.title('QED vs SA')
        plt.tight_layout()
        plt.show()


# 5. Histograms for application-specific metrics
if homo_lumo is not None and len(homo_lumo) > 0:
    cleaned = [x for x in homo_lumo if x is not None]
    if cleaned:
        plt.figure()
        plt.hist(cleaned, bins=30)
        plt.xlabel('HOMO–LUMO Gap')
        plt.ylabel('Frequency')
        plt.title('Distribution of HOMO–LUMO Gaps')
        plt.tight_layout()
        plt.show()

if lewis_desc is not None and len(lewis_desc) > 0:
    cleaned = [x for x in lewis_desc if x is not None]
    if cleaned:
        plt.figure()
        plt.hist(cleaned, bins=30)
        plt.xlabel('Lewis Descriptor')
        plt.ylabel('Frequency')
        plt.title('Distribution of Lewis Descriptors')
        plt.tight_layout()
        plt.show()


In [ ]:
def sample_and_decode(generator, latent_dim, dataset, device, n_samples=1):
    generator = generator.to(device)
    generator.eval()
    decoded_smiles = []

    with torch.no_grad():
        for _ in range(n_samples):
            z = torch.randn(1, latent_dim, device=device)
            logits = generator(z)  # [1, seq_len, vocab_size]
            logits = logits.squeeze(0)  # [seq_len, vocab_size]

            # Take argmax token index at each position
            token_indices = logits.argmax(dim=-1).tolist()
            tokens = [dataset.idx_to_token.get(idx, '') for idx in token_indices if idx != 0]
            selfies_str = ''.join(tokens)

            try:
                smiles_str = selfies.decoder(selfies_str)
                mol = Chem.MolFromSmiles(smiles_str)
                if mol:
                    decoded_smiles.append(smiles_str)
            except:
                continue

    return decoded_smiles

In [ ]:
sampled_smiles = sample_and_decode(generator, latent_dim, dataset, device, n_samples=100)
scores = [combined_flp_reward(smi) for smi in sampled_smiles]

print("Top 5 scored FLP candidates:")
top_5 = sorted(zip(sampled_smiles, scores), key=lambda x: -x[1])[:5]
for smi, score in top_5:
    print(f"{smi} --> Score: {score:.3f}")